# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print("Dataset Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List the record sets with their IDs
print("Available Record Sets:\n")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']}: {rs['name'] if 'name' in rs else ''}")

# For each record set, print available fields and their IDs
for rs in dataset.metadata.record_sets:
    print(f"\nRecord Set @id: {rs['@id']}, Name: {rs.get('name','N/A')}")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  - Field @id: {field['@id']} ({field.get('name', 'N/A')})   type: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references use their `@id`.

We will extract all record sets found in the metadata.


In [ ]:
# Extract data from each record set using their @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    # dataset.records returns a generator of dict
    try:
        records = list(dataset.records(record_set=record_set_id))
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")
        records = []
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nColumns for '{record_set_id}':")
        print(dataframes[record_set_id].columns.tolist())
        print(f"First records of '{record_set_id}':")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for '{record_set_id}'.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** Replace `record_set_id`, `numeric_field_id`, and `group_field_id` below with a valid record set and numeric field as observed above. For demonstration, let's select the first available record set and numeric field if present.

In [ ]:
import numpy as np

# If any dataframes were loaded, select the first for demonstration
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Try to auto-detect a numeric field (int/float columns)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try to coerce any columns to numeric that might be stored as object
        for col in df.columns:
            try:
                # Skip columns that could be entirely null/empty
                if df[col].dropna().size > 0:
                    df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        # Filtering
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to find a categorical field (object column with low cardinality)
        group_fields = df.select_dtypes(include=['object', 'category']).nunique()
        group_fields = [f for f, n in group_fields.items() if n < 20 and f != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields found in selected record set for EDA.")
else:
    print("No data frames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if EDA selected a DataFrame and numeric field
if dataframes and 'numeric_field_id' in locals():
    # Plot histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If we found a group_field_id and it has not too many unique values, show boxplot
    if 'group_field_id' in locals() and df[group_field_id].nunique() < 20:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"'{numeric_field_id}' grouped by '{group_field_id}'")
        plt.show()
else:
    print("No suitable numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library. We reviewed the available data structures, loaded tabular data with appropriate `@id` references, performed basic filtering and normalization, and visualized data distributions. Please refer to the dataset documentation and Croissant schema for detailed information on each record set and field. This approach can be adapted for deeper domain analyses, model development, or integration with additional omics data.